In [ ]:
import json
import gradio as gr
from openai import OpenAI

In [ ]:
MODEL = 'llama3.2:3b'
openai = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

In [ ]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [ ]:
ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
  print(f"Tool called for city {destination_city}")
  price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
  return f"The price of a ticket to {destination_city} is {price}"

# get_ticket_price("London")

In [ ]:
price_function = {
  "name": "get_ticket_price",
  "description": "Get the price of a return ticket to the destination city.",
  "parameters": {
    "type": "object",
    "properties": {
      "destination_city": {
        "type": "string",
        "description": "The city that the customer wants to travel to",
      },
    },
    "required": ["destination_city"],
    "additionalProperties": False
  }
}

tools = [{"type": "function", "function": price_function}]

In [ ]:
def handle_tool_calls(message):
  responses = []
  for tool_call in message.tool_calls:
    if tool_call.function.name == "get_ticket_price":
      arguments = json.loads(tool_call.function.arguments)
      city = arguments.get('destination_city')
      price_details = get_ticket_price(city)
      responses.append({
          "role": "tool",
          "content": price_details,
          "tool_call_id": tool_call.id
      })

  return responses

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()